## Ejercicio "Taller 01" para limpieza de datos con Pandas.

¡Hola, profe! Este Python Notebook contiene el proceso que se llevó a cabo para limpiar una base de datos en .csv.

La base de datos contiene un análisis de Google Ads para un curso de análisis de datos. El nombre del archivo es "GoogleAds_DataAnalytics_Sales_Uncleaned.csv"

Las 3 preguntas que se responderán al limpiar la base de datos serán:
## - ¿Cuál fue el total de ingresos generados?
## - ¿Cuál fue el retorno de inversión (ROI) de las campañas?
## - ¿Qué dispositivo genera más conversiones?

Importamos las librerías "Pandas" y "Numpy"

In [49]:
import pandas as pd
import numpy as np

Importamos el archivo .csv para su lectura y mostramos las 5 primeras filas:

In [50]:
datos_taller = pd.read_csv('GoogleAds_DataAnalytics_Sales_Uncleaned.csv')
datos_taller.head()

,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course
2,A1002,Data Anlytics Corse,90.0,4544.0,$203.66,26.0,9.0,NaN,$1624,2024/11/16,hyderabad,Desktop,data analitics online
3,A1003,Data Analytcis Course,142.0,3185.0,$237.66,17.0,6.0,NaN,$1225,2024-11-26,HYDERABAD,tablet,data anaytics training
4,A1004,Data Analytics Corse,156.0,3361.0,$195.9,30.0,8.0,NaN,$1091,2024-11-22,hyderabad,desktop,online data analytic


Creamos un nuevo índice con la columna "Ad_ID" y guardamos:

In [51]:
datos_taller.set_index('Ad_ID', inplace=True)

Con el comando .head, vemos que hay colunas con datos sin unificar, tanto escritos como numéricos y valores vacíos o NaN. Realizamos la unificación de los datos:

La IA sugirió un paso para poder contar los valores únicos y unificar los datos que estén escritos de una manera distinta:

In [ ]:
for col in ['Campaign_Name', 'Location', 'Device']:
    print(f"\n--- {col} ---")
    print(datos_taller[col].value_counts(dropna=False))

Con este código, se crea un bucle donde muestra los valores "col" en las columnas "Campaign_Name", "Location" y "Device".
Después va a mostrar (print) los datos de cada columna como título entre 6 guiones y crea un salto de línea para que se vea ordenado.
Finalmente, selecciona la columna y muestra los valores únicos que hay y cuenta cuantos valores únicos hay, incluyendo los valores vacíos (NaN).

Vamos a convertir todos los valores que hayan en la columna #Campaign_Name" a texto (str), todo a minúsculas para que sea más ordenado (str.lower) y eliminar espacios (str.strip):

In [53]:
for col in ['Campaign_Name', 'Location', 'Device', 'Keyword']:
    datos_taller[col] = datos_taller[col].astype(str).str.lower().str.strip()

Corremos nuevamente el comando para contar los valores únicos en "Campaign name":

In [ ]:
datos_taller['Campaign_Name'].value_counts()

Mediante el comando .replace vamos a reemplazar los datos mal escritos en la columna "Campaign_Name" por el valor que tiene más datos 'data analytics course':

In [55]:
datos_taller['Campaign_Name'] = datos_taller['Campaign_Name'].replace({
    'data anlytics corse': 'data analytics course',
    'data analytcis course': 'data analytics course',
    'data analytics corse': 'data analytics course',
    'dataanalyticscourse': 'data analytics course'
})

Hacemos el mismo proceso con la columna "Location" para dejar los datos unificados para la provincia de Hyderabad en India:

In [56]:
datos_taller['Location'] = datos_taller['Location'].replace({
    'hyderabad': 'Hyderabad',
    'hyderbad': 'Hyderabad',
    'hydrebad': 'Hyderabad'
})

Repetimos el comando condicional que creamos para mostrar que los datos estén unificados y los errores se hayan corregido:

In [ ]:
for col in ['Campaign_Name', 'Location', 'Device']:
    print(f"\n--- {col} ---")
    print(datos_taller[col].value_counts(dropna=False))

Ya teniendo los datos en las columnas de texto unificados y en concordancia, verificamos las columnas con fechas y otros datos numéricos:

In [ ]:
datos_taller.dtypes

Verificamos los 5 primeros datos de la columna "Conversion rate", para que nos muestre decimales y no enteros:

In [ ]:
datos_taller['Conversion Rate'].head()

Vamos a cambiar los datos "NaN" a 0. Esto con el fin de no borrar las filas que no hayan sido convertidas y dejar el 0 como valor de no conversión:

In [60]:
datos_taller['Conversion Rate'] = datos_taller['Conversion Rate'].fillna(0)

Con el comando datos_taller.dtypes, vimos que las columnas de Cost y Sale_Amount aparecen como string, porque contienen símbolos de $, y necesitan ser float    .

Limpiamos la columna Cost indicando a Pandas que el $ no es un regex (regular expresion) y que lo deje vacío para que se convierta en número (.astype(float)):

In [61]:
datos_taller['Cost'] = (
    datos_taller['Cost']
    .str.replace('$', '', regex=False)
    .astype(float)
)

Igual con Sale_Amount:

In [62]:
datos_taller['Sale_Amount'] = (
    datos_taller['Sale_Amount']
    .str.replace('$', '', regex=False)
    .astype(float)
)

En el siguiente paso, vamos a dejar la fecha en un formato unificado de fecha, ya que hay filas mezcladas entre el año primero, el día primero y separada con distintos separadores:

In [63]:
datos_taller['Ad_Date'] = pd.to_datetime(
    datos_taller['Ad_Date'].astype(str).str.replace('/', '-', regex=False),
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

Cambiamos los separadores de "/" a "-" y unificamos las fechas poniendo el día primero. Con el errors='coerce' convertimos los vacíos en fechas sin que nos arroje un posible error. Además, como hay múltiples formatos de fecha en la columna Ad_Date, se usa el pd.to_datetime() con format='mixed' para convertir las fechas sin que Pandas se confunda y entienda las diferencias de fechas.

Con la base lista y limpia, podemos pasar a responder las preguntas:

## Pregunta 1: ¿Cuál fue el total de ingresos generados?

En esta sección se calcula el total de ingresos generados por las campañas publicitarias utilizando la columna "Sale_Amount" del dataset.

In [ ]:
# Cálculo de ingresos totales
total_ingresos = datos_taller['Sale_Amount'].sum()

# Mostrar los ingresos totales
print(f"El total de ingresos generados es: ${total_ingresos:,.2f}")
# el ,.2f formatea el número con dos decimales y , como separador de miles

El total de ingresos generados es: $3,688,173.00


## Pregunta 2: ¿Cuál fue el retorno de inversión (ROI) de las campañas?

El ROI (Return on Investment) compara los ingresos generados frente a los costos invertidos.

In [ ]:
# Cálculo del ROI
ingresos = datos_taller['Sale_Amount'].sum()
costos = datos_taller['Cost'].sum()

roi = (ingresos - costos) / costos

# Mostrar el ROI
print(f"El ROI de las campañas es: {roi:.2%}")
# el.2% formatea el número como porcentaje con dos decimales

El ROI de las campañas es: 585.06%


## Pregunta 3: ¿Qué dispositivo genera más conversiones?

En esta sección se analiza el número total de conversiones agrupadas por tipo de dispositivo (Device) para identificar cuál tiene mejor desempeño.

In [ ]:
# Conversiones por dispositivo
conversiones_device = datos_taller.groupby('Device')['Conversions'].sum()

# Mostrar conversiones por dispositivo
print("Conversiones por dispositivo:")
print(conversiones_device)

# Dispositivo con más conversiones
mejor_device = conversiones_device.idxmax()
valor_max = conversiones_device.max()

print(f"\nEl dispositivo con más conversiones es '{mejor_device}' con un total de {valor_max:.0f} conversiones.")
# el .0f formatea el número sin decimales.
# el idxmax() devuelve el índice del valor máximo, que en este caso es el nombre del dispositivo. Y el max() devuelve el valor máximo de conversiones para ese dispositivo.

Conversiones por dispositivo:
Device
desktop    5632.0
mobile     5645.0
tablet     5190.0
Name: Conversions, dtype: float64

El dispositivo con más conversiones es 'mobile' con un total de 5645 conversiones.
